# pycd48 Tutorial - CD48 Coincidence Counter

This notebook provides an interactive tutorial for using the **pycd48** library to control the Red Dog Physics CD48 Coincidence Counter.

## What is the CD48?

The CD48 is a professional 4-channel coincidence counter designed for advanced physics experiments:
- 🌌 Cosmic ray detection and muon lifetime measurements
- 🔬 Quantum optics experiments (entanglement, Bell inequalities)
- ⚛️ Nuclear physics and particle detection
- 📊 Multi-detector correlation studies

### Key Specifications
- **4 input channels** (A, B, C, D) with BNC connectors
- **8 configurable counters** for singles and coincidences
- **~25 ns coincidence window** for precise timing
- **Programmable trigger threshold** (0-4.08V)
- **Selectable input impedance** (50Ω or High-Z)

## Prerequisites

Before running this notebook, ensure you have:
1. A CD48 device connected via USB
2. The pycd48 library installed: `uv sync` or `pip install -e .`
3. Input signals connected to the CD48 (or run in demo mode with no signals)

## 1. Setup and Imports

First, let's import the necessary libraries:

In [ ]:
import time
import numpy as np
import matplotlib.pyplot as plt
from pycd48 import CD48, DataLogger

# Configure matplotlib for notebook display
%matplotlib inline
plt.style.use('seaborn-v0_8-darkgrid')

print("✓ Imports successful")

## 2. Connect to the CD48

The `CD48` class will automatically detect the device on your system. You can also specify the port manually if needed:
- Windows: `CD48(port='COM3')`
- Linux/Mac: `CD48(port='/dev/ttyUSB0')`

In [ ]:
# Connect to the CD48
cd48 = CD48()  # Auto-detects port

print(f"✓ Connected to CD48")
print(f"Firmware version: {cd48.get_version()}")

## 3. Device Information

Let's check the current device settings and test the LEDs:

In [ ]:
# Display current settings
print("Current CD48 Settings:")
print("=" * 50)
print(cd48.get_settings(human_readable=True))

In [ ]:
# Test the LEDs (all LEDs will light up for 1 second)
print("Testing LEDs... (watch your CD48!)")
cd48.test_leds()
print("✓ LED test complete")

## 4. Configure Channels

The CD48 has 8 counters (0-7) that can be configured to count:
- **Singles**: Events on individual channels (A, B, C, or D)
- **2-fold coincidences**: Simultaneous events on 2 channels (e.g., A AND B)
- **3-fold coincidences**: Simultaneous events on 3 channels (e.g., A AND B AND C)
- **4-fold coincidences**: Simultaneous events on all 4 channels

Let's configure a typical setup:

In [ ]:
# Configure counters
print("Configuring counters...")

# Singles counting
cd48.set_channel(0, A=1, B=0, C=0, D=0)  # Counter 0: Channel A singles
cd48.set_channel(1, A=0, B=1, C=0, D=0)  # Counter 1: Channel B singles
cd48.set_channel(2, A=0, B=0, C=1, D=0)  # Counter 2: Channel C singles
cd48.set_channel(3, A=0, B=0, C=0, D=1)  # Counter 3: Channel D singles

# 2-fold coincidences
cd48.set_channel(4, A=1, B=1, C=0, D=0)  # Counter 4: A-B coincidences
cd48.set_channel(5, A=1, B=0, C=1, D=0)  # Counter 5: A-C coincidences

# 3-fold coincidences
cd48.set_channel(6, A=1, B=1, C=1, D=0)  # Counter 6: A-B-C triple coincidences

# 4-fold coincidences
cd48.set_channel(7, A=1, B=1, C=1, D=1)  # Counter 7: A-B-C-D quad coincidences

# Set trigger level (adjust based on your signal amplitude)
cd48.set_trigger_level(0.5)  # 0.5V threshold
print("  Trigger level set to 0.5V")

# Set input impedance to 50 Ohm (use set_impedance_highz() for high impedance)
cd48.set_impedance_50ohm()
print("  Input impedance set to 50Ω")

print("\n✓ Configuration complete")

## 5. Simple Counting Measurement

Let's perform a simple counting measurement over a fixed interval:

In [ ]:
# Clear counters and start measurement
duration = 10  # seconds
print(f"Starting {duration}-second measurement...")

cd48.clear_counts()
time.sleep(duration)

# Read results
data = cd48.get_counts(human_readable=False)
counts = data["counts"]
overflow = data["overflow"]

# Display results
print(f"\n{'Channel':<25} {'Counts':>10} {'Rate (Hz)':>12}")
print("="*50)
labels = [
    "A singles",
    "B singles",
    "C singles",
    "D singles",
    "A-B coincidences",
    "A-C coincidences",
    "A-B-C triple coinc.",
    "A-B-C-D quad coinc."
]

for i, label in enumerate(labels):
    rate = counts[i] / duration
    print(f"{label:<25} {counts[i]:>10,} {rate:>12.2f}")

if overflow:
    print(f"\n⚠️  WARNING: Counter overflow detected! (0x{overflow:02X})")
else:
    print("\n✓ No overflow detected")

## 6. Continuous Data Collection

Now let's collect data continuously and visualize it in real-time. This is useful for monitoring stability and seeing fluctuations:

In [ ]:
# Collection parameters
total_duration = 60  # Total measurement time in seconds
interval = 2         # Sampling interval in seconds

# Data storage
times = []
counts_A = []
counts_B = []
coincidences_AB = []

print(f"Collecting data for {total_duration} seconds (interval: {interval}s)...")
print(f"{'Time(s)':<8} {'Ch A':>8} {'Ch B':>8} {'A-B Coinc':>10}")
print("="*40)

start_time = time.time()

while time.time() - start_time < total_duration:
    cd48.clear_counts()
    time.sleep(interval)
    
    data = cd48.get_counts(human_readable=False)
    
    current_time = time.time() - start_time
    times.append(current_time)
    counts_A.append(data["counts"][0])
    counts_B.append(data["counts"][1])
    coincidences_AB.append(data["counts"][4])
    
    print(f"{current_time:>7.1f}  {counts_A[-1]:>8,}  {counts_B[-1]:>8,}  {coincidences_AB[-1]:>10,}")

print("\n✓ Data collection complete")

## 7. Data Analysis and Visualization

Let's analyze the collected data and create visualizations:

In [ ]:
# Convert to numpy arrays for analysis
times = np.array(times)
counts_A = np.array(counts_A)
counts_B = np.array(counts_B)
coincidences_AB = np.array(coincidences_AB)

# Calculate statistics
print("Statistical Summary:")
print("="*50)
print(f"Channel A:       {counts_A.mean():>8.2f} ± {counts_A.std():>6.2f} counts/{interval}s")
print(f"Channel B:       {counts_B.mean():>8.2f} ± {counts_B.std():>6.2f} counts/{interval}s")
print(f"Coincidences AB: {coincidences_AB.mean():>8.2f} ± {coincidences_AB.std():>6.2f} counts/{interval}s")

# Calculate count rates (per second)
rate_A = counts_A.mean() / interval
rate_B = counts_B.mean() / interval
rate_AB = coincidences_AB.mean() / interval

print(f"\nCount Rates:")
print(f"Channel A:       {rate_A:>8.2f} Hz")
print(f"Channel B:       {rate_B:>8.2f} Hz")
print(f"Coincidences AB: {rate_AB:>8.2f} Hz")

### Accidental Coincidence Analysis

When two uncorrelated particle detectors are operating, random coincidences can occur. The expected accidental coincidence rate is:

$$R_{acc} = 2 \tau \cdot R_A \cdot R_B$$

where:
- $\tau$ = coincidence window (~25 ns for CD48)
- $R_A$, $R_B$ = singles rates on channels A and B

In [ ]:
# Calculate accidental coincidence rate
tau = 25e-9  # 25 nanoseconds coincidence window
accidental_rate = 2 * tau * rate_A * rate_B
true_coincidence_rate = rate_AB - accidental_rate

print("Coincidence Analysis:")
print("="*50)
print(f"Measured coincidence rate: {rate_AB:>10.4f} Hz")
print(f"Expected accidental rate:  {accidental_rate:>10.4f} Hz")
print(f"True coincidence rate:     {true_coincidence_rate:>10.4f} Hz")

if true_coincidence_rate > 0:
    accidental_fraction = (accidental_rate / rate_AB) * 100
    print(f"\nAccidental coincidences:   {accidental_fraction:>10.2f}% of total")
else:
    print("\nNote: All coincidences appear to be accidental (uncorrelated sources)")

### Time Series Visualization

In [ ]:
# Create comprehensive plots
fig, axes = plt.subplots(3, 1, figsize=(14, 10))

# Plot 1: Time series of all channels
axes[0].plot(times, counts_A, 'o-', label='Channel A', alpha=0.7, markersize=6)
axes[0].plot(times, counts_B, 's-', label='Channel B', alpha=0.7, markersize=6)
axes[0].plot(times, coincidences_AB, '^-', label='A-B Coincidences', alpha=0.7, markersize=6)
axes[0].set_xlabel('Time (s)', fontsize=11)
axes[0].set_ylabel(f'Counts per {interval}s interval', fontsize=11)
axes[0].set_title('CD48 Count Rates Over Time', fontsize=13, fontweight='bold')
axes[0].legend(loc='best', fontsize=10)
axes[0].grid(True, alpha=0.3)

# Plot 2: Distribution histograms
bins = min(15, len(counts_A) // 2)  # Adaptive bin count
axes[1].hist(counts_A, bins=bins, alpha=0.6, label='Channel A', edgecolor='black')
axes[1].hist(counts_B, bins=bins, alpha=0.6, label='Channel B', edgecolor='black')
axes[1].hist(coincidences_AB, bins=bins, alpha=0.6, label='A-B Coincidences', edgecolor='black')
axes[1].set_xlabel(f'Counts per {interval}s interval', fontsize=11)
axes[1].set_ylabel('Frequency', fontsize=11)
axes[1].set_title('Distribution of Count Rates', fontsize=13, fontweight='bold')
axes[1].legend(loc='best', fontsize=10)
axes[1].grid(True, alpha=0.3, axis='y')

# Plot 3: Coincidence to singles ratio
ratio = coincidences_AB / (counts_A + 1)  # Add 1 to avoid division by zero
axes[2].plot(times, ratio * 100, 'o-', color='purple', alpha=0.7, markersize=6)
axes[2].set_xlabel('Time (s)', fontsize=11)
axes[2].set_ylabel('Coincidence/Singles Ratio (%)', fontsize=11)
axes[2].set_title('A-B Coincidence to A Singles Ratio', fontsize=13, fontweight='bold')
axes[2].grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig('cd48_analysis.png', dpi=150, bbox_inches='tight')
print("\n✓ Plot saved as 'cd48_analysis.png'")
plt.show()

## 8. Advanced Features

### DAC Output Control

The CD48 has a DAC output (0-4.08V) that can be used to control external equipment:

In [ ]:
# Example: Set DAC voltage
dac_voltage = 2.0  # 2.0V output
cd48.set_dac_voltage(dac_voltage)
print(f"✓ DAC output set to {dac_voltage:.2f}V")

# You can use this for:
# - Controlling voltage-variable components
# - Triggering external equipment
# - Automated voltage sweeps

### Data Logging to CSV

The `DataLogger` class makes it easy to save data for long-term measurements:

In [ ]:
# Example: Create a data logger (commented out to avoid creating files)
# Uncomment to use:

# logger = DataLogger('cd48_log.csv')
# 
# # Log data periodically
# for i in range(10):
#     cd48.clear_counts()
#     time.sleep(1)
#     data = cd48.get_counts(human_readable=False)
#     logger.log(data['counts'])
# 
# logger.close()
# print("✓ Data logged to 'cd48_log.csv'")

print("See examples/data_logger.py for full logging example")

### Overflow Detection

Counters 0-6 are 24-bit (max: 16,777,215) and counter 7 is 16-bit (max: 65,535). Check for overflow:

In [ ]:
# Check overflow status
overflow = cd48.get_overflow()

if overflow:
    print(f"⚠️  Overflow detected on counters: 0x{overflow:02X}")
    for i in range(8):
        if overflow & (1 << i):
            print(f"   - Counter {i} overflowed")
else:
    print("✓ No overflow detected")

print("\nTip: Use shorter measurement intervals if overflow occurs frequently")

## 9. Cleanup

Always close the connection when done:

In [ ]:
# Close connection
cd48.close()
print("✓ CD48 connection closed")

## Summary and Next Steps

### What you learned:
- ✅ Connecting to the CD48
- ✅ Configuring channels for singles and coincidence counting
- ✅ Performing measurements and reading counts
- ✅ Continuous data collection and analysis
- ✅ Calculating accidental coincidences
- ✅ Visualizing data with matplotlib
- ✅ Using advanced features (DAC, overflow detection)

### Next steps:
1. **Explore more examples** in the `examples/` directory:
   - `cosmic_ray_telescope.py` - Multi-detector cosmic ray setup
   - `calibrate_trigger.py` - Automatic trigger level optimization
   - `realtime_monitor.py` - Real-time monitoring with auto-repeat mode

2. **Build your own experiments**:
   - Cosmic ray flux measurements
   - Quantum entanglement demonstrations
   - Gamma-gamma coincidence spectroscopy

3. **Read the documentation**:
   - [README.md](../README.md) - Full API reference
   - [CLAUDE.md](../CLAUDE.md) - Development guide

### Tips for successful measurements:
- 📌 Adjust trigger level based on your signal amplitude
- 📌 Use 50Ω impedance for fast signals, high-Z for slow/large signals
- 📌 Account for accidental coincidences in correlation studies
- 📌 Monitor for overflow in high-rate applications
- 📌 Use appropriate measurement intervals for your count rates

---

**Happy experimenting! 🔬**